In [21]:
import pandas as pd
import numpy as np

transactions = pd.read_csv('trans_labelled_clean.csv')
cards = pd.read_csv('cards_data_south_africa.csv')
users = pd.read_csv('user_data_south_africa.csv')

trans_merged = pd.merge(transactions, cards, left_on='card_id', right_on='id')
trans_merged = pd.merge(trans_merged, users, left_on='client_id_x', right_on='id')
trans_merged.drop(columns=['id_y', 'client_id_y', 'id'])
trans_merged = trans_merged.rename(columns={
    'id_x': 'id',
    'client_id_x': 'user_id',
})

In [22]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Production‑grade feature engineering for fraud detection.
    All target‑dependent statistics are fitted on training data only.
    Temporal features use only past information (no look‑ahead).
    """

    def __init__(self,
                 high_value_quantile: float = 0.95,
                 iqr_k: float = 1.5,
                 hour_zscore_thresh: float = 2.0,
                 robust_zscore_thresh: float = 2.0,
                 city_change_min_sec: int = 60,
                 city_change_max_sec: int = 3600,
                 rare_merchant_threshold: int = 3,
                 default_time_since_last: float = 9999.0,
                 rolling_windows: list = None):
        self.high_value_quantile = high_value_quantile
        self.iqr_k = iqr_k
        self.hour_zscore_thresh = hour_zscore_thresh
        self.robust_zscore_thresh = robust_zscore_thresh
        self.city_change_min_sec = city_change_min_sec
        self.city_change_max_sec = city_change_max_sec
        self.rare_merchant_threshold = rare_merchant_threshold
        self.default_time_since_last = default_time_since_last
        self.rolling_windows = rolling_windows or ['24h', '7D']

        # Attributes filled in fit()
        self.high_value_threshold_ = None
        self.merchant_risk_ = None          # map: merchant_id -> fraud rate
        self.mcc_risk_ = None              # map: mcc -> fraud rate
        self.card_brand_risk_ = None       # map: card_brand -> fraud rate
        self.mcc_avg_amount_ = None        # map: mcc -> mean(abs_amount)
        self.user_hour_stats_ = None       # DataFrame: user_id, hour_mean, hour_std
        self.user_hour_robust_ = None      # DataFrame: user_id, hour_median, hour_mad
        self.user_amount_quantiles_ = None # DataFrame: user_id, q1, q3, iqr
        self.city_region_map_ = None       # mapping from city to region
        self.required_columns_ = None      # for input validation

    def fit(self, X: pd.DataFrame, y: pd.Series = None):
        """
        Compute all global and per‑group statistics that depend on the target
        or require the entire training set. Store them as attributes.
        """
        X = X.copy()
        y = y.copy()

        # --- Basic validation and preparation ---
        required = ['amount', 'date', 'user_id', 'merchant_id', 'mcc',
                    'card_brand', 'yearly_income', 'acct_open_date',
                    'year_pin_last_changed', 'merchant_city', 'merchant_state',
                    'address']
        missing = [col for col in required if col not in X.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        self.required_columns_ = required

        # Convert date to datetime
        X['date'] = pd.to_datetime(X['date'])

        # --- Absolute amount (always used) ---
        X['abs_amount'] = X['amount'].abs()

        # --- 1. Global high value threshold ---
        self.high_value_threshold_ = X['abs_amount'].quantile(self.high_value_quantile)

        # --- 5. MCC average amount ---
        self.mcc_avg_amount_ = X.groupby('mcc')['abs_amount'].mean().to_dict()

        # --- 6. User hour statistics (mean & std) ---
        self.user_hour_stats_ = (
            X.groupby('user_id')['transaction_hour']
            .agg(hour_mean='mean', hour_std='std')
            .reset_index()
        )

        # --- 7. Robust user hour statistics (median & MAD) ---
        def mad(s):
            return np.median(np.abs(s - np.median(s)))

        self.user_hour_robust_ = (
            X.groupby('user_id')['transaction_hour']
            .agg(hour_median='median', hour_mad=mad)
            .reset_index()
        )

        # --- 8. User amount quantiles for IQR outlier detection (static) ---
        user_quantiles = (
            X.groupby('user_id')['abs_amount']
            .quantile([0.25, 0.75])
            .unstack()
            .reset_index()
            .rename(columns={0.25: 'q1', 0.75: 'q3'})
        )
        user_quantiles['iqr'] = user_quantiles['q3'] - user_quantiles['q1']
        self.user_amount_quantiles_ = user_quantiles

        # --- 9. City → Region mapping (clean and standardised) ---
        city_to_region = {
            "cape town": "western cape",
            "pretoria": "gauteng",
            "johannesburg": "gauteng",
            "durban": "kwazulu-natal",
            "pietermaritzburg": "kwazulu-natal",
            "gqeberha": "eastern cape",
            "east london": "eastern cape",
            "bloemfontein": "free state",
        }
        self.city_region_map_ = city_to_region

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Apply feature engineering using **only** past information and
        pre‑computed statistics from fit().
        """
        # --- 1. Copy & input validation ---
        df = X.copy()
        if self.required_columns_ is None:
            raise RuntimeError("fit() must be called before transform()")
        missing = [col for col in self.required_columns_ if col not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # --- 2. Core conversions & base features ---
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

        # --- Amount ---
        df['is_negative_amount'] = (df['amount'] < 0).astype(int)
        df['abs_amount'] = df['amount'].abs()
        # Keep original 'amount'? Usually dropped. We'll drop after using it.
        # We'll keep it for now and drop leakage columns at the end.

        # --- Temporal features ---
        df['transaction_hour'] = df['date'].dt.hour
        df['transaction_dayofweek'] = df['date'].dt.dayofweek
        df['is_weekend'] = df['transaction_dayofweek'].isin([5, 6]).astype(int)
        df['is_night'] = ((df['transaction_hour'] >= 22) | (df['transaction_hour'] <= 6)).astype(int)

        # --- 3. Apply pre‑fitted risk scores ---
        df['merchant_risk_score'] = df['merchant_id'].map(self.merchant_risk_)
        # Fill missing merchants with global fraud rate (stored as attribute? we don't have it)
        # Use the mean of the mapped values as fallback
        default_risk = np.nanmean(list(self.merchant_risk_.values())) if self.merchant_risk_ else 0.0
        df['merchant_risk_score'] = df['merchant_risk_score'].fillna(default_risk)

        df['mcc_risk_score'] = df['mcc'].map(self.mcc_risk_)
        default_mcc_risk = np.nanmean(list(self.mcc_risk_.values())) if self.mcc_risk_ else 0.0
        df['mcc_risk_score'] = df['mcc_risk_score'].fillna(default_mcc_risk)

        df['card_brand_risk'] = df['card_brand'].map(self.card_brand_risk_)
        default_brand_risk = np.nanmean(list(self.card_brand_risk_.values())) if self.card_brand_risk_ else 0.0
        df['card_brand_risk'] = df['card_brand_risk'].fillna(default_brand_risk)

        df['mcc_avg_amount'] = df['mcc'].map(self.mcc_avg_amount_)
        # Fill with global mean of abs_amount from training? We'll compute on the fly from this batch as fallback.
        global_avg_amount = df['abs_amount'].mean()
        df['mcc_avg_amount'] = df['mcc_avg_amount'].fillna(global_avg_amount)

        # --- 4. Time since last transaction (hours) ---
        df['time_since_last_txn'] = (
            df.groupby('user_id')['date']
            .diff()
            .dt.total_seconds()
            .div(3600)
            .fillna(self.default_time_since_last)
        )

        # --- 5. Expanding user statistics (using only past transactions) ---
        g_user = df.groupby('user_id')
        df['user_avg_amount'] = (
            g_user['abs_amount']
            .transform(lambda x: x.expanding().mean().shift())
            .fillna(df['abs_amount'])      # first transaction -> current amount
        )
        df['user_std_amount'] = (
            g_user['abs_amount']
            .transform(lambda x: x.expanding().std().shift())
            .fillna(0)
        )
        df['amount_to_avg_ratio'] = df['abs_amount'] / (df['user_avg_amount'] + 1e-6)
        # Clip extreme ratios
        df['amount_to_avg_ratio'] = df['amount_to_avg_ratio'].clip(0, 10).fillna(1.0)

        # --- 6. High value flag (using pre‑fitted global threshold) ---
        df['is_high_value'] = (df['abs_amount'] > self.high_value_threshold_).astype(int)

        # --- 7. Rolling window features (time‑based, shift to avoid look‑ahead) ---
        for window in self.rolling_windows:
            # Transaction count in last window
            count_col = f'txn_count_{window.replace("h","h").replace("D","d")}'
            df[count_col] = (
                g_user.rolling(window, on='date')['abs_amount']
                .count()
                .shift(1)
                .reset_index(drop=True)
                .fillna(0)
            )
            # Sum of amounts in last window
            sum_col = f'amt_sum_{window.replace("h","h").replace("D","d")}'
            df[sum_col] = (
                g_user.rolling(window, on='date')['abs_amount']
                .sum()
                .shift(1)
                .reset_index(drop=True)
                .fillna(0)
            )
            # Average amount in last window
            avg_col = f'avg_amt_{window.replace("h","h").replace("D","d")}'
            df[avg_col] = df[sum_col] / (df[count_col] + 1)

        # --- 8. Merchant / city visit history (using cumcount with shift) ---
        df['prev_visits_to_merchant'] = (
            df.groupby(['user_id', 'merchant_id']).cumcount().shift(1).fillna(0)
        )
        df['is_first_merchant_visit'] = (df['prev_visits_to_merchant'] == 0).astype(int)
        df['is_rare_merchant'] = (df['prev_visits_to_merchant'] < self.rare_merchant_threshold).astype(int)

        df['prev_visits_to_city'] = (
            df.groupby(['user_id', 'merchant_city']).cumcount().shift(1).fillna(0)
        )
        df['is_first_city_visit'] = (df['prev_visits_to_city'] == 0).astype(int)

        # --- 9. Location matching (clean & standardised) ---
        # Extract user city from address
        df['user_city'] = df['address'].str.split(',').str[1].str.strip()
        # Normalisation function
        def _normalize_city(s):
            if pd.isna(s):
                return ''
            return (str(s).strip().lower()
                    .replace('.', '')
                    .replace('-', ' ')
                    .replace(',', ''))
        df['user_city'] = df['user_city'].apply(_normalize_city)
        df['merchant_city'] = df['merchant_city'].apply(_normalize_city)
        df['user_region'] = df['user_city'].map(self.city_region_map_).fillna('unknown')
        df['merchant_state'] = df['merchant_state'].str.lower().str.strip().fillna('')

        # Binary indicators
        df['is_same_city'] = (
            (df['user_city'] != '') &
            (df['merchant_city'] != '') &
            (df['user_city'] == df['merchant_city'])
        ).astype(int)

        df['is_same_region'] = (
            (df['user_region'] != '') &
            (df['merchant_state'] != '') &
            (df['user_region'] == df['merchant_state'])
        ).astype(int)

        # --- 10. City change & suspicious speed ---
        df['prev_city'] = df.groupby('user_id')['merchant_city'].shift(1)
        df['city_changed'] = (
            (df['merchant_city'] != df['prev_city']) &
            df['prev_city'].notna()
        ).astype(int)

        df['time_diff_sec'] = (
            df.groupby('user_id')['date'].diff().dt.total_seconds()
        )
        df['city_change_fast'] = (
            (df['city_changed'] == 1) &
            df['time_diff_sec'].between(self.city_change_min_sec, self.city_change_max_sec)
        ).astype(int)

        # --- 11. Card age & years since PIN change ---
        df['acct_open_date'] = pd.to_datetime(df['acct_open_date'])
        df['card_age_days'] = (df['date'] - df['acct_open_date']).dt.days

        df['years_since_pin_change'] = df['date'].dt.year - df['year_pin_last_changed']

        # --- 12. Income‑based features (safe handling of zero income) ---
        df['yearly_income'] = df['yearly_income'].replace(0, np.nan)
        df['amt_to_income_ratio'] = df['abs_amount'] / df['yearly_income']
        df['amt_to_income_ratio'] = df['amt_to_income_ratio'].fillna(0).clip(0, 1e6)
        df['log_amt_to_income_ratio'] = np.log1p(df['amt_to_income_ratio'])
        df['income_ratio_outlier'] = (df['amt_to_income_ratio'] > 0.1).astype(int)

        # --- 13. User hour anomaly scores (using pre‑fitted stats) ---
        # Merge pre‑computed hour stats
        df = df.merge(self.user_hour_stats_, on='user_id', how='left')
        df['hour_mean'] = df['hour_mean'].fillna(df['transaction_hour'].mean())
        df['hour_std'] = df['hour_std'].fillna(df['transaction_hour'].std())
        df['unusual_hour'] = (
            (df['transaction_hour'] - df['hour_mean']).abs() >
            (self.hour_zscore_thresh * df['hour_std'])
        ).astype(int)

        # Robust z‑score (median + MAD)
        df = df.merge(self.user_hour_robust_, on='user_id', how='left')
        df['hour_median'] = df['hour_median'].fillna(df['transaction_hour'].median())
        df['hour_mad'] = df['hour_mad'].fillna(1.0)   # avoid division by zero
        df['robust_z_hour'] = (
            (df['transaction_hour'] - df['hour_median']) / (1.4826 * df['hour_mad'] + 1e-6)
        )

        # --- 14. IQR outlier detection (static per‑user from training) ---
        df = df.merge(self.user_amount_quantiles_, on='user_id', how='left')
        df['q1'] = df['q1'].fillna(df['abs_amount'].quantile(0.25))
        df['q3'] = df['q3'].fillna(df['abs_amount'].quantile(0.75))
        df['iqr'] = df['q3'] - df['q1']
        lower_iqr = df['q1'] - self.iqr_k * df['iqr']
        upper_iqr = df['q3'] + self.iqr_k * df['iqr']
        df['is_iqr_outlier'] = (
            (df['abs_amount'] < lower_iqr) | (df['abs_amount'] > upper_iqr)
        ).astype(int)
        # For users with zero IQR, flag if amount differs from Q1
        df['is_iqr_outlier'] = np.where(
            df['iqr'] == 0,
            (df['abs_amount'] != df['q1']).astype(int),
            df['is_iqr_outlier']
        )

        # --- 15. Z‑score outlier (using expanding mean/std with shift) ---
        df['user_amount_mean'] = (
            g_user['abs_amount'].transform(lambda x: x.expanding().mean().shift())
            .fillna(df['abs_amount'])
        )
        df['user_amount_std'] = (
            g_user['abs_amount'].transform(lambda x: x.expanding().std().shift())
            .fillna(0)
        )
        df['amount_zscore'] = (
            (df['abs_amount'] - df['user_amount_mean']) /
            (df['user_amount_std'] + 1e-6)
        )
        df['is_z_outlier'] = (df['amount_zscore'] > 2.5).astype(int)

        # --- 16. MCC amount deviation ---
        df['mcc_amount_deviation'] = (
            (df['abs_amount'] - df['mcc_avg_amount']) /
            (df['mcc_avg_amount'] + 1e-6)
        )

        # --- 17. Drop temporary & leakage columns ---
        # Columns created only for intermediate calculations
        temp_cols = [
            'prev_city', 'time_diff_sec', 'hour_mean', 'hour_std',
            'hour_median', 'hour_mad', 'q1', 'q3', 'iqr',
            'user_amount_mean', 'user_amount_std', 'mcc_avg_amount',
            'prev_visits_to_merchant', 'prev_visits_to_city'
        ]
        df.drop(columns=[c for c in temp_cols if c in df.columns], inplace=True, errors='ignore')

        # Known leakage columns (should be removed from final feature set)
        leakage_cols = [
            'id', 'id.1', 'id_y', 'user_id', 'card_id', 'client_id_y',
            'card_number', 'cvv', 'expires', 'address', 'zip',
            'description', 'errors', 'gender', 'amount'      # raw amount replaced by abs_amount
        ]
        df.drop(columns=[c for c in leakage_cols if c in df.columns], inplace=True, errors='ignore')

        # Round selected float columns for readability
        float_cols_to_round = [
            'time_since_last_txn', 'user_avg_amount', 'user_std_amount',
            'amount_to_avg_ratio', 'merchant_risk_score', 'mcc_risk_score',
            'amt_to_income_ratio', 'log_amt_to_income_ratio', 'robust_z_hour',
            'amount_zscore', 'mcc_amount_deviation', 'card_brand_risk'
        ]
        for col in float_cols_to_round:
            if col in df.columns:
                df[col] = df[col].round(3)

        # Ensure no infinite values remain
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        # Fill remaining NaNs with sensible defaults (0 for counts, mean for ratios, etc.)
        # This step is optional; you may prefer to leave them for the model to handle.
        df.fillna({
            'user_std_amount': 0,
            'amount_to_avg_ratio': 1.0,
            'txn_count_24h': 0,
            'txn_count_7D': 0,
            'amt_sum_24h': 0,
            'avg_amt_24h': df['abs_amount'].mean(),
            'robust_z_hour': 0,
            'amount_zscore': 0,
            'mcc_amount_deviation': 0,
            'is_rare_merchant': 0,
            'is_first_merchant_visit': 0,
            'is_first_city_visit': 0,
        }, inplace=True)

        return df

In [23]:
target = "is_fraud"

X = trans_merged.drop(columns=[target])
y = trans_merged[target]

numeric_cols = X.select_dtypes(include=['int', 'float']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel

feature_engineer = FraudFeatureEngineer()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), numeric_cols),

        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

feature_selector = SelectFromModel(
    XGBClassifier(n_estimators=200, eval_metric='aucpr'),
    threshold="median"
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    scale_pos_weight=3000,
    eval_metric='aucpr'
)

fraud_pipeline = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessing', preprocessor),
    ('feature_selection', feature_selector),
    ('model', model)
])


In [25]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

In [26]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

lgbm = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=255,
    min_data_in_leaf=30,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5,
    scale_pos_weight=pos_weight,
    max_bin=255,           # 500 is unnecessary & memory-heavy
    force_row_wise=True,
    n_jobs=-1,
    random_state=42
)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,           # 10 is often too deep for fraud
    subsample=0.7,
    colsample_bytree=0.9,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",   # IMPORTANT for fraud
    gamma=1,
    min_child_weight=5,    # stronger regularization
    reg_alpha=1.0,
    reg_lambda=2.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)


#rf = RandomForestClassifier(
#    n_estimators=150,
#    max_depth=10,
#    class_weight="balanced",
#    random_state=42
#)
base_models = [
    ('lgbm', lgbm),
    ('xgb_model', xgb_model),
#    ('rf', rf)
]

meta_learner = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=0.1,              # strong regularization
        penalty="l2",
        max_iter=3000,
        solver="lbfgs",
        n_jobs=-1
    ))
])


stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv = TimeSeriesSplit(n_splits=4),
    n_jobs=1,
    passthrough=False,
    stack_method="predict_proba"
)

In [27]:
from sklearn.pipeline import Pipeline

fraud_pipeline = Pipeline([
    ('feature_engineering', FraudFeatureEngineer()),
    ('preprocessing', preprocessor),
    ('feature_selection', feature_selector),
    ('model', stacked_model)
])

In [28]:
print(X_train.columns)

Index(['id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors',
       'description', 'id_y', 'client_id_y', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'id', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards'],
      dtype='object')


In [29]:
fraud_pipeline.fit(X_train, y_train)

KeyError: 'Column not found: transaction_hour'

In [8]:
import joblib

joblib.dump(fraud_pipeline, "fraud_pipeline.pkl")

['fraud_pipeline.pkl']

In [9]:
best_threshold = 0.99
joblib.dump(best_threshold, "fraud_threshold_v1.pkl")

['fraud_threshold_v1.pkl']

In [11]:
import joblib
import pandas as pd

# Load
model = joblib.load("fraud_pipeline.pkl")
threshold = joblib.load("fraud_threshold_v1.pkl")

# Load completely raw data (same schema as training input)
new_data = pd.read_csv("trans_unlabelled_clean.csv")

# Predict
probs = model.predict_proba(new_data)[:, 1]
preds = (probs >= threshold).astype(int)

RuntimeError: fit() must be called before transform()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_test, probs))
print("PR-AUC:", average_precision_score(y_test, probs))
print(confusion_matrix(y_test, preds))
